# Artificial Intelligence Laboratory (26UAMPC204)
**Sanjivani University | Department of Artificial Intelligence & Data Science**  
**For DSY (Direct Second Year) - AI & ML Students**

Prepared by **Dr. S. Satyanarayana, Ph.D., PDF (AI)**, Professor, Department of AI & DS, Sanjivani University

This notebook contains the complete, tested Python program for each of the 10 experiments, with its input and output. It needs Python 3.10 or later and only the standard library. Run a cell with **Shift + Enter**; each experiment is independent, so the cells can be run in any order.

| Exp. | Title | CO |
|---|---|---|
| 1 | Uninformed Search: BFS and DFS for the Water-Jug Problem | CO204.1 |
| 2 | A* Search with an Admissible Heuristic for the 8-Puzzle | CO204.2 |
| 3 | Hill Climbing and Best-First Search for a State-Space Problem | CO204.2 |
| 4 | Minimax with Alpha-Beta Pruning (Tic-Tac-Toe) | CO204.2 |
| 5 | Constraint Satisfaction by Backtracking: N-Queens and Map Colouring | CO204.2 |
| 6 | Facts, Rules and Logical Inference (Prolog style in Python) | CO204.3 |
| 7 | Unification and Resolution for First-Order Logic | CO204.3 |
| 8 | Bayes' Theorem and a Simple Bayesian Network for Diagnosis | CO204.4 |
| 9 | Decision-Tree Classifier (ID3) and the Rules Learned | CO204.5 |
| 10 | Mini-Project: Rule-Based Expert System and a Basic Chatbot | CO204.5 |


---
## Experiment 1: Uninformed Search: BFS and DFS for the Water-Jug Problem
**Course Outcome:** CO204.1

**Aim:** To formulate the Water-Jug problem as a state-space search problem and solve it using Breadth-First Search (BFS) and Depth-First Search (DFS).

**Input:** `CAP_A = 4`, `CAP_B = 3`, `TARGET = 2`, start state `(0, 0)`. Try `5, 3, 4` and `6, 4, 3` (the second has no solution because gcd(6, 4) = 2 does not divide 3).


In [1]:
"""Experiment 1: Uninformed Search (BFS and DFS) - Water Jug Problem."""
from collections import deque

CAP_A, CAP_B, TARGET = 4, 3, 2       # jug capacities and goal amount


def successors(state):
    """Return every (next_state, action) reachable from 'state'."""
    a, b = state
    pour_ab = min(a, CAP_B - b)      # litres that can move A -> B
    pour_ba = min(b, CAP_A - a)      # litres that can move B -> A
    moves = [
        ((CAP_A, b), "Fill jug A"),
        ((a, CAP_B), "Fill jug B"),
        ((0, b), "Empty jug A"),
        ((a, 0), "Empty jug B"),
        ((a - pour_ab, b + pour_ab), "Pour A -> B"),
        ((a + pour_ba, b - pour_ba), "Pour B -> A"),
    ]
    return [(s, act) for s, act in moves if s != state]


def is_goal(state):
    return state[0] == TARGET


def build_path(parent, state):
    """Walk back through the parent links to recover the solution."""
    path = []
    while parent[state] is not None:
        previous, action = parent[state]
        path.append((action, state))
        state = previous
    return path[::-1]


def search(start, strategy):
    """Generic graph search. BFS uses a FIFO queue, DFS a LIFO stack."""
    frontier = deque([start])
    parent = {start: None}           # also acts as the visited set
    expanded = 0
    while frontier:
        state = frontier.popleft() if strategy == "BFS" else frontier.pop()
        expanded += 1
        if is_goal(state):
            return build_path(parent, state), expanded
        for nxt, action in successors(state):
            if nxt not in parent:
                parent[nxt] = (state, action)
                frontier.append(nxt)
    return None, expanded


def report(strategy, start):
    path, expanded = search(start, strategy)
    print(f"\n===== {strategy} =====")
    if path is None:
        print("No solution exists.")
        return
    print(f"Step  0: {'Start':<13} -> {start}")
    for i, (action, state) in enumerate(path, 1):
        print(f"Step {i:2d}: {action:<13} -> {state}")
    print(f"Solution length : {len(path)} moves")
    print(f"Nodes expanded  : {expanded}")


if __name__ == "__main__":
    print(f"Water Jug Problem: A = {CAP_A} L, B = {CAP_B} L, "
          f"goal = {TARGET} L in jug A")
    for name in ("BFS", "DFS"):
        report(name, (0, 0))

Water Jug Problem: A = 4 L, B = 3 L, goal = 2 L in jug A

===== BFS =====
Step  0: Start         -> (0, 0)
Step  1: Fill jug A    -> (4, 0)
Step  2: Pour A -> B   -> (1, 3)
Step  3: Empty jug B   -> (1, 0)
Step  4: Pour A -> B   -> (0, 1)
Step  5: Fill jug A    -> (4, 1)
Step  6: Pour A -> B   -> (2, 3)
Solution length : 6 moves
Nodes expanded  : 13

===== DFS =====
Step  0: Start         -> (0, 0)
Step  1: Fill jug B    -> (0, 3)
Step  2: Pour B -> A   -> (3, 0)
Step  3: Fill jug B    -> (3, 3)
Step  4: Pour B -> A   -> (4, 2)
Step  5: Empty jug A   -> (0, 2)
Step  6: Pour B -> A   -> (2, 0)
Solution length : 6 moves
Nodes expanded  : 7


---
## Experiment 2: A* Search with an Admissible Heuristic for the 8-Puzzle
**Course Outcome:** CO204.2

**Aim:** To implement A* for the 8-puzzle with the admissible heuristics *misplaced tiles* and *Manhattan distance*, and compare the nodes they expand. Evaluation function: $f(n) = g(n) + h(n)$.

**Input:** Start `(2,8,3, 1,6,4, 7,0,5)`, goal `(1,2,3, 8,0,4, 7,6,5)` (`0` is the blank). A harder start `(2,1,6, 4,0,8, 7,5,3)` is used to compare the heuristics.


In [2]:
"""Experiment 2: A* Search with admissible heuristics - 8-Puzzle."""
import heapq

GOAL = (1, 2, 3,
        8, 0, 4,
        7, 6, 5)                      # 0 denotes the blank tile
GOAL_POS = {tile: divmod(i, 3) for i, tile in enumerate(GOAL)}


def h_misplaced(state):
    """h1: number of tiles that are not in their goal cell."""
    return sum(1 for i, t in enumerate(state) if t != 0 and t != GOAL[i])


def h_manhattan(state):
    """h2: sum of horizontal + vertical distances of tiles from goal."""
    total = 0
    for i, tile in enumerate(state):
        if tile != 0:
            r, c = divmod(i, 3)
            gr, gc = GOAL_POS[tile]
            total += abs(r - gr) + abs(c - gc)
    return total


def neighbours(state):
    """Slide the blank Up, Down, Left or Right (each move costs 1)."""
    z = state.index(0)
    r, c = divmod(z, 3)
    for dr, dc, name in ((-1, 0, "Up"), (1, 0, "Down"),
                         (0, -1, "Left"), (0, 1, "Right")):
        nr, nc = r + dr, c + dc
        if 0 <= nr < 3 and 0 <= nc < 3:
            s = list(state)
            s[z], s[nr * 3 + nc] = s[nr * 3 + nc], s[z]
            yield tuple(s), name


def inversions(state):
    tiles = [t for t in state if t != 0]
    return sum(tiles[i] > tiles[j]
               for i in range(len(tiles)) for j in range(i + 1, len(tiles)))


def solvable(start):
    """On a 3x3 board a state can reach GOAL iff parities match."""
    return inversions(start) % 2 == inversions(GOAL) % 2


def astar(start, h):
    open_list = [(h(start), 0, start)]          # entries are (f, g, state)
    g_cost = {start: 0}
    parent = {start: None}
    expanded = 0
    while open_list:
        f, g, state = heapq.heappop(open_list)
        if g > g_cost[state]:
            continue                            # stale queue entry
        expanded += 1
        if state == GOAL:
            path = []
            while parent[state] is not None:
                state, move = parent[state][0], parent[state][1]
                path.append(move)
            return path[::-1], expanded
        for nxt, move in neighbours(state):
            new_g = g + 1
            if new_g < g_cost.get(nxt, float("inf")):
                g_cost[nxt] = new_g
                parent[nxt] = (state, move)
                heapq.heappush(open_list, (new_g + h(nxt), new_g, nxt))
    return None, expanded


def show(state):
    for r in range(3):
        print("   " + " ".join("_" if t == 0 else str(t)
                               for t in state[3 * r:3 * r + 3]))


def apply(state, move):
    return next(s for s, m in neighbours(state) if m == move)


if __name__ == "__main__":
    start = (2, 8, 3,
             1, 6, 4,
             7, 0, 5)
    print("Solvable:", solvable(start))
    path, expanded = astar(start, h_manhattan)
    state = start
    print(f"g=0 h={h_manhattan(state)} f={h_manhattan(state)}  (start)")
    show(state)
    for g, move in enumerate(path, 1):
        state = apply(state, move)
        h = h_manhattan(state)
        print(f"g={g} h={h} f={g + h}  (blank moved {move})")
        show(state)
    print("Optimal moves :", " -> ".join(path))
    print("Path cost     :", len(path))

    hard = (2, 1, 6,
            4, 0, 8,
            7, 5, 3)
    print("\nHeuristic comparison on a harder instance", hard)
    for name, h in (("h1 misplaced", h_misplaced), ("h2 Manhattan", h_manhattan)):
        p, n = astar(hard, h)
        print(f"  {name:<13}: cost = {len(p):2d}, nodes expanded = {n}")

Solvable: True
g=0 h=5 f=5  (start)
   2 8 3
   1 6 4
   7 _ 5
g=1 h=4 f=5  (blank moved Up)
   2 8 3
   1 _ 4
   7 6 5
g=2 h=3 f=5  (blank moved Up)
   2 _ 3
   1 8 4
   7 6 5
g=3 h=2 f=5  (blank moved Left)
   _ 2 3
   1 8 4
   7 6 5
g=4 h=1 f=5  (blank moved Down)
   1 2 3
   _ 8 4
   7 6 5
g=5 h=0 f=5  (blank moved Right)
   1 2 3
   8 _ 4
   7 6 5
Optimal moves : Up -> Up -> Left -> Down -> Right
Path cost     : 5

Heuristic comparison on a harder instance (2, 1, 6, 4, 0, 8, 7, 5, 3)
  h1 misplaced : cost = 18, nodes expanded = 1747
  h2 Manhattan : cost = 18, nodes expanded = 229


---
## Experiment 3: Hill Climbing and Best-First Search for a State-Space Problem
**Course Outcome:** CO204.2

**Aim:** To implement (a) steepest-ascent Hill Climbing with random restart for 8-Queens and (b) Greedy Best-First Search on the Romania road map.

**Input:** Part A: `N = 8`, random seed `7`. Part B: road map `GRAPH`, straight-line distances `H_SLD`, start `Arad`, goal `Bucharest`.


In [3]:
"""Experiment 3: Hill Climbing (8-Queens) and Greedy Best-First Search."""
import heapq
import random

N = 8


# ---------- Part A: steepest-ascent hill climbing with random restart ------
def conflicts(state):
    """h(n): number of pairs of queens attacking each other.
    state[c] = row of the queen placed in column c."""
    return sum(1 for a in range(N) for b in range(a + 1, N)
               if state[a] == state[b] or abs(state[a] - state[b]) == b - a)


def best_neighbour(state):
    """Move one queen inside its column; return the lowest-h neighbour."""
    best, best_h = state, conflicts(state)
    for col in range(N):
        for row in range(N):
            if row != state[col]:
                cand = state[:col] + (row,) + state[col + 1:]
                h = conflicts(cand)
                if h < best_h:
                    best, best_h = cand, h
    return best, best_h


def hill_climb(state):
    steps = 0
    while True:
        nxt, h = best_neighbour(state)
        if nxt == state:                 # no better neighbour: peak reached
            return state, conflicts(state), steps
        state, steps = nxt, steps + 1


def random_restart(seed=7, max_restarts=50):
    rng = random.Random(seed)
    for attempt in range(1, max_restarts + 1):
        start = tuple(rng.randrange(N) for _ in range(N))
        final, h, steps = hill_climb(start)
        status = "GLOBAL optimum" if h == 0 else "stuck (local optimum)"
        print(f"Restart {attempt}: start h = {conflicts(start):2d} -> "
              f"final h = {h} after {steps} steps  [{status}]")
        if h == 0:
            return final
    return None


def show_board(state):
    for row in range(N):
        print("   " + " ".join("Q" if state[c] == row else "." for c in range(N)))


# ---------- Part B: greedy best-first search on the Romania road map -------
GRAPH = {
    "Arad": {"Zerind": 75, "Sibiu": 140, "Timisoara": 118},
    "Zerind": {"Arad": 75, "Oradea": 71},
    "Oradea": {"Zerind": 71, "Sibiu": 151},
    "Timisoara": {"Arad": 118, "Lugoj": 111},
    "Lugoj": {"Timisoara": 111, "Mehadia": 70},
    "Mehadia": {"Lugoj": 70, "Drobeta": 75},
    "Drobeta": {"Mehadia": 75, "Craiova": 120},
    "Sibiu": {"Arad": 140, "Oradea": 151, "Fagaras": 99, "Rimnicu": 80},
    "Fagaras": {"Sibiu": 99, "Bucharest": 211},
    "Rimnicu": {"Sibiu": 80, "Pitesti": 97, "Craiova": 146},
    "Craiova": {"Drobeta": 120, "Rimnicu": 146, "Pitesti": 138},
    "Pitesti": {"Rimnicu": 97, "Craiova": 138, "Bucharest": 101},
    "Bucharest": {"Fagaras": 211, "Pitesti": 101},
}
H_SLD = {"Arad": 366, "Zerind": 374, "Oradea": 380, "Timisoara": 329,
         "Lugoj": 244, "Mehadia": 241, "Drobeta": 242, "Sibiu": 253,
         "Fagaras": 176, "Rimnicu": 193, "Craiova": 160, "Pitesti": 100,
         "Bucharest": 0}


def greedy_best_first(start, goal):
    frontier = [(H_SLD[start], start)]           # ordered by h(n) only
    parent = {start: None}
    while frontier:
        h, city = heapq.heappop(frontier)
        print(f"   expand {city:<10} h = {h}")
        if city == goal:
            path = []
            while city is not None:
                path.append(city)
                city = parent[city]
            return path[::-1]
        for nxt in GRAPH[city]:
            if nxt not in parent:
                parent[nxt] = city
                heapq.heappush(frontier, (H_SLD[nxt], nxt))
    return None


if __name__ == "__main__":
    print("PART A: Hill climbing on the 8-Queens problem")
    solution = random_restart()
    print("Solution (row of queen in each column):", solution)
    show_board(solution)

    print("\nPART B: Greedy best-first search, Arad -> Bucharest")
    path = greedy_best_first("Arad", "Bucharest")
    cost = sum(GRAPH[a][b] for a, b in zip(path, path[1:]))
    print("Path :", " -> ".join(path))
    print("Cost :", cost, "km  (optimal is 418 km via Rimnicu and Pitesti)")

PART A: Hill climbing on the 8-Queens problem
Restart 1: start h =  6 -> final h = 1 after 3 steps  [stuck (local optimum)]
Restart 2: start h =  9 -> final h = 1 after 3 steps  [stuck (local optimum)]
Restart 3: start h = 10 -> final h = 0 after 5 steps  [GLOBAL optimum]
Solution (row of queen in each column): (2, 4, 1, 7, 0, 6, 3, 5)
   . . . . Q . . .
   . . Q . . . . .
   Q . . . . . . .
   . . . . . . Q .
   . Q . . . . . .
   . . . . . . . Q
   . . . . . Q . .
   . . . Q . . . .

PART B: Greedy best-first search, Arad -> Bucharest
   expand Arad       h = 366
   expand Sibiu      h = 253
   expand Fagaras    h = 176
   expand Bucharest  h = 0
Path : Arad -> Sibiu -> Fagaras -> Bucharest
Cost : 450 km  (optimal is 418 km via Rimnicu and Pitesti)


---
## Experiment 4: Minimax with Alpha-Beta Pruning (Tic-Tac-Toe)
**Course Outcome:** CO204.2

**Aim:** To implement Minimax and its Alpha-Beta pruned version for Tic-Tac-Toe and compare the number of game-tree nodes visited.

**Input:** Cells are numbered 0 to 8, row by row. Three fixed test cases: an empty board, the position `X O X / . O . / . . .` with X to move, and a full computer-versus-computer game. Set `HUMAN_PLAYS = True` to type the moves of O yourself.


In [4]:
"""Experiment 4: Minimax with Alpha-Beta pruning - Tic-Tac-Toe."""
import math

LINES = [(0, 1, 2), (3, 4, 5), (6, 7, 8),      # rows
         (0, 3, 6), (1, 4, 7), (2, 5, 8),      # columns
         (0, 4, 8), (2, 4, 6)]                 # diagonals
HUMAN_PLAYS = False        # set True to play as O against the computer
nodes = 0                  # counts the game-tree nodes visited


def winner(board):
    for a, b, c in LINES:
        if board[a] != " " and board[a] == board[b] == board[c]:
            return board[a]
    return "Draw" if " " not in board else None


def utility(result, depth):
    """+ve for X (MAX), -ve for O (MIN); quicker wins score higher."""
    if result == "X":
        return 10 - depth
    if result == "O":
        return depth - 10
    return 0


def minimax(board, player, depth=0):
    global nodes
    nodes += 1
    result = winner(board)
    if result:
        return utility(result, depth), None
    best_move = None
    best = -math.inf if player == "X" else math.inf
    for cell in range(9):
        if board[cell] == " ":
            board[cell] = player
            score, _ = minimax(board, "O" if player == "X" else "X", depth + 1)
            board[cell] = " "
            if (player == "X" and score > best) or \
               (player == "O" and score < best):
                best, best_move = score, cell
    return best, best_move


def alphabeta(board, player, depth=0, alpha=-math.inf, beta=math.inf):
    global nodes
    nodes += 1
    result = winner(board)
    if result:
        return utility(result, depth), None
    best_move = None
    if player == "X":                               # MAX node
        best = -math.inf
        for cell in range(9):
            if board[cell] == " ":
                board[cell] = "X"
                score, _ = alphabeta(board, "O", depth + 1, alpha, beta)
                board[cell] = " "
                if score > best:
                    best, best_move = score, cell
                alpha = max(alpha, best)
                if alpha >= beta:                   # beta cut-off
                    break
    else:                                           # MIN node
        best = math.inf
        for cell in range(9):
            if board[cell] == " ":
                board[cell] = "O"
                score, _ = alphabeta(board, "X", depth + 1, alpha, beta)
                board[cell] = " "
                if score < best:
                    best, best_move = score, cell
                beta = min(beta, best)
                if alpha >= beta:                   # alpha cut-off
                    break
    return best, best_move


def show(board):
    for r in range(3):
        print("   " + " | ".join(board[3 * r:3 * r + 3]))
        if r < 2:
            print("   --+---+--")


def count_nodes(algorithm, board, player):
    global nodes
    nodes = 0
    value, move = algorithm(board, player)
    return value, move, nodes


def play():
    board, player, history = [" "] * 9, "X", []
    while winner(board) is None:
        if HUMAN_PLAYS and player == "O":
            move = int(input("Your move (0-8): "))
        else:
            _, move = alphabeta(board, player)
        board[move] = player
        history.append(f"{player}{move}")
        player = "O" if player == "X" else "X"
    print("Moves (player+cell):", " ".join(history))
    show(board)
    print("Result:", winner(board))


if __name__ == "__main__":
    empty = [" "] * 9
    print("1) Search effort for the first move on an empty board")
    for name, algo in (("Minimax", minimax), ("Alpha-Beta", alphabeta)):
        value, move, n = count_nodes(algo, empty[:], "X")
        print(f"   {name:<10}: value = {value}, best cell = {move}, "
              f"nodes visited = {n}")

    print("\n2) X to move in a tactical position")
    position = ["X", "O", "X",
                " ", "O", " ",
                " ", " ", " "]
    show(position)
    value, move, n = count_nodes(alphabeta, position[:], "X")
    print(f"   Alpha-Beta chooses cell {move} (value {value}, {n} nodes): "
          "X must block at cell 7")

    print("\n3) Computer (X) versus computer (O), both playing optimally")
    play()

1) Search effort for the first move on an empty board
   Minimax   : value = 0, best cell = 0, nodes visited = 549946
   Alpha-Beta: value = 0, best cell = 0, nodes visited = 20866

2) X to move in a tactical position
   X | O | X
   --+---+--
     | O |  
   --+---+--
     |   |  
   Alpha-Beta chooses cell 7 (value 0, 110 nodes): X must block at cell 7

3) Computer (X) versus computer (O), both playing optimally
Moves (player+cell): X0 O4 X1 O2 X6 O3 X5 O7 X8
   X | X | O
   --+---+--
   O | O | X
   --+---+--
   X | O | X
Result: Draw


---
## Experiment 5: Constraint Satisfaction by Backtracking: N-Queens and Map Colouring
**Course Outcome:** CO204.2

**Aim:** To model N-Queens and Map Colouring as Constraint Satisfaction Problems and solve them using backtracking search.

**Input:** Part A: board sizes N = 4 to 8. Part B: adjacency list `NEIGHBOURS` of Australia and colours `Red, Green, Blue`.


In [5]:
"""Experiment 5: Constraint Satisfaction by Backtracking.
Part A: N-Queens.   Part B: Map colouring of Australia."""


# ---------------- Part A: N-Queens ----------------
def safe(queens, row, col):
    """queens[r] = column of the queen already placed in row r."""
    for r, c in enumerate(queens):
        if c == col or abs(c - col) == row - r:
            return False
    return True


def n_queens(n):
    """Return (first solution, number of solutions, number of backtracks)."""
    solutions, stats = [], {"backtracks": 0}

    def place(queens):
        row = len(queens)
        if row == n:
            solutions.append(tuple(queens))
            return
        for col in range(n):
            if safe(queens, row, col):
                queens.append(col)
                place(queens)
                queens.pop()                 # undo the choice: BACKTRACK
                stats["backtracks"] += 1

    place([])
    first = solutions[0] if solutions else None
    return first, len(solutions), stats["backtracks"]


def show_board(solution):
    n = len(solution)
    for r in range(n):
        print("   " + " ".join("Q" if solution[r] == c else "." for c in range(n)))


# ---------------- Part B: Map colouring ----------------
NEIGHBOURS = {
    "WA": ["NT", "SA"],
    "NT": ["WA", "SA", "Q"],
    "SA": ["WA", "NT", "Q", "NSW", "V"],
    "Q": ["NT", "SA", "NSW"],
    "NSW": ["Q", "SA", "V"],
    "V": ["SA", "NSW"],
    "T": [],
}
COLOURS = ["Red", "Green", "Blue"]


def consistent(region, colour, assignment):
    return all(assignment.get(nb) != colour for nb in NEIGHBOURS[region])


def select_unassigned(assignment):
    """Degree heuristic: pick the unassigned region with most neighbours."""
    free = [r for r in NEIGHBOURS if r not in assignment]
    return max(free, key=lambda r: len(NEIGHBOURS[r]))


def colour_map(assignment, trace):
    if len(assignment) == len(NEIGHBOURS):
        return assignment
    region = select_unassigned(assignment)
    for colour in COLOURS:
        if consistent(region, colour, assignment):
            assignment[region] = colour
            trace.append(f"assign {region:<3} = {colour}")
            result = colour_map(assignment, trace)
            if result:
                return result
            del assignment[region]           # BACKTRACK
            trace.append(f"undo   {region}")
    return None


if __name__ == "__main__":
    print("PART A: N-Queens by backtracking")
    print("   N  solutions  backtracks")
    for n in range(4, 9):
        _, count, back = n_queens(n)
        print(f"  {n:2d}  {count:9d}  {back:10d}")
    first, _, _ = n_queens(8)
    print("First solution for N = 8 (column of queen in each row):", first)
    show_board(first)

    print("\nPART B: Map colouring of Australia with 3 colours")
    trace = []
    result = colour_map({}, trace)
    for line in trace:
        print("  ", line)
    print("Final assignment:")
    for region in NEIGHBOURS:
        print(f"   {region:<3} -> {result[region]}")
    ok = all(result[r] != result[nb] for r in NEIGHBOURS for nb in NEIGHBOURS[r])
    print("All constraints satisfied:", ok)

PART A: N-Queens by backtracking
   N  solutions  backtracks
   4          2          16
   5         10          53
   6          4         152
   7         40         551
   8         92        2056
First solution for N = 8 (column of queen in each row): (0, 4, 7, 5, 2, 6, 1, 3)
   Q . . . . . . .
   . . . . Q . . .
   . . . . . . . Q
   . . . . . Q . .
   . . Q . . . . .
   . . . . . . Q .
   . Q . . . . . .
   . . . Q . . . .

PART B: Map colouring of Australia with 3 colours
   assign SA  = Red
   assign NT  = Green
   assign Q   = Blue
   assign NSW = Green
   assign WA  = Blue
   assign V   = Blue
   assign T   = Red
Final assignment:
   WA  -> Blue
   NT  -> Green
   SA  -> Red
   Q   -> Blue
   NSW -> Green
   V   -> Blue
   T   -> Red
All constraints satisfied: True


---
## Experiment 6: Facts, Rules and Logical Inference (Prolog style in Python)
**Course Outcome:** CO204.3

**Aim:** To represent knowledge as facts and rules (Horn clauses) and answer queries by forward-chaining inference. Names starting with a capital letter are variables, as in Prolog.

**Input:** 16 base facts (`male`, `female`, `parent`), 6 rules, and 7 queries listed at the bottom of the program.


In [6]:
"""Experiment 6: Facts, rules and logical inference (Prolog style) in Python.
Convention (same as Prolog): names starting with an UPPER-case letter are
variables, lower-case names are constants."""

FACTS = {
    ("male", "dasharath"), ("male", "ram"), ("male", "lakshman"),
    ("male", "lav"), ("male", "kush"),
    ("female", "kausalya"), ("female", "sumitra"), ("female", "sita"),
    ("parent", "dasharath", "ram"), ("parent", "kausalya", "ram"),
    ("parent", "dasharath", "lakshman"), ("parent", "sumitra", "lakshman"),
    ("parent", "ram", "lav"), ("parent", "sita", "lav"),
    ("parent", "ram", "kush"), ("parent", "sita", "kush"),
}

# Each rule is (head, [body literals]) and reads:  head :- body1, body2, ...
RULES = [
    (("father", "X", "Y"), [("parent", "X", "Y"), ("male", "X")]),
    (("mother", "X", "Y"), [("parent", "X", "Y"), ("female", "X")]),
    (("grandparent", "X", "Z"), [("parent", "X", "Y"), ("parent", "Y", "Z")]),
    (("sibling", "X", "Y"), [("parent", "P", "X"), ("parent", "P", "Y"),
                             ("neq", "X", "Y")]),
    (("ancestor", "X", "Y"), [("parent", "X", "Y")]),
    (("ancestor", "X", "Z"), [("parent", "X", "Y"), ("ancestor", "Y", "Z")]),
]


def is_var(term):
    return term[0].isupper()


def match(pattern, fact, env):
    """Try to match a literal against a ground fact; extend bindings."""
    if pattern[0] != fact[0] or len(pattern) != len(fact):
        return None
    env = dict(env)
    for p, f in zip(pattern[1:], fact[1:]):
        if is_var(p):
            if env.setdefault(p, f) != f:
                return None
        elif p != f:
            return None
    return env


def solve(body, facts, env):
    """Yield every binding that satisfies all literals of the body."""
    if not body:
        yield env
        return
    first, rest = body[0], body[1:]
    if first[0] == "neq":                       # built-in: X \= Y
        a, b = (env.get(t, t) for t in first[1:])
        if a != b:
            yield from solve(rest, facts, env)
        return
    for fact in facts:
        new_env = match(first, fact, env)
        if new_env is not None:
            yield from solve(rest, facts, new_env)


def forward_chain(facts, rules):
    """Apply Modus Ponens repeatedly until no new fact can be derived."""
    facts, rounds = set(facts), 0
    while True:
        new = set()
        for head, body in rules:
            for env in solve(body, facts, {}):
                derived = tuple(env.get(t, t) for t in head)
                if derived not in facts:
                    new.add(derived)
        if not new:
            return facts, rounds
        facts |= new
        rounds += 1
        print(f"   round {rounds}: {len(new)} new facts derived")


def ask(query, kb):
    """Answer a query; variables in the query are reported as bindings."""
    answers = sorted({tuple(sorted(env.items()))
                      for env in solve([query], kb, {})})
    text = f"?- {query[0]}({', '.join(query[1:])})."
    if not any(is_var(t) for t in query[1:]):
        print(f"{text:<34} {'true' if answers else 'false'}")
    elif not answers:
        print(f"{text:<34} false")
    else:
        shown = "; ".join(", ".join(f"{v} = {c}" for v, c in a) for a in answers)
        print(f"{text:<34} {shown}")


if __name__ == "__main__":
    print(f"Base facts: {len(FACTS)}, rules: {len(RULES)}")
    kb, rounds = forward_chain(FACTS, RULES)
    print(f"Fixed point after {rounds} rounds: {len(kb)} facts in total\n")
    for q in [("father", "dasharath", "ram"),
              ("mother", "sita", "ram"),
              ("father", "X", "lav"),
              ("grandparent", "X", "kush"),
              ("sibling", "lav", "Y"),
              ("ancestor", "dasharath", "D"),
              ("sibling", "ram", "S")]:
        ask(q, kb)

Base facts: 16, rules: 6
   round 1: 24 new facts derived
   round 2: 4 new facts derived
Fixed point after 2 rounds: 44 facts in total

?- father(dasharath, ram).         true
?- mother(sita, ram).              false
?- father(X, lav).                 X = ram
?- grandparent(X, kush).           X = dasharath; X = kausalya
?- sibling(lav, Y).                Y = kush
?- ancestor(dasharath, D).         D = kush; D = lakshman; D = lav; D = ram
?- sibling(ram, S).                S = lakshman


---
## Experiment 7: Unification and Resolution for First-Order Logic
**Course Outcome:** CO204.3

**Aim:** To implement the unification algorithm (with occurs check) and prove a first-order logic statement by resolution refutation.

**Input:** Part A: six pairs of expressions (variables start with `?`). Part B: the *Colonel West* knowledge base in clause form (C1 to C8) and the goal `Criminal(West)`. In the output `~` means NOT and `v` means OR.


In [7]:
"""Experiment 7: Unification and Resolution for First-Order Logic.
Terms : variable  -> string starting with '?'      e.g. '?x'
        constant  -> any other string              e.g. 'John'
        compound  -> tuple (functor, arg1, ...)    e.g. ('Knows', 'John', '?x')
Literal: (sign, atom) where sign is True for positive, False for negated."""


def is_var(t):
    return isinstance(t, str) and t.startswith("?")


def substitute(t, theta):
    """Apply substitution theta to term t."""
    if is_var(t):
        return substitute(theta[t], theta) if t in theta else t
    if isinstance(t, tuple):
        return tuple(substitute(a, theta) for a in t)
    return t


def occurs(var, t, theta):
    """Occurs check: does var appear inside t?  (blocks x = f(x))"""
    t = substitute(t, theta)
    if var == t:
        return True
    return isinstance(t, tuple) and any(occurs(var, a, theta) for a in t)


def unify(x, y, theta=None):
    """Return the most general unifier of x and y, or None on failure."""
    theta = {} if theta is None else theta
    x, y = substitute(x, theta), substitute(y, theta)
    if x == y:
        return theta
    if is_var(x):
        return None if occurs(x, y, theta) else {**theta, x: y}
    if is_var(y):
        return unify(y, x, theta)
    if isinstance(x, tuple) and isinstance(y, tuple) and len(x) == len(y):
        for a, b in zip(x, y):
            theta = unify(a, b, theta)
            if theta is None:
                return None
        return theta
    return None


def show(t):
    if isinstance(t, tuple):
        return f"{t[0]}({', '.join(show(a) for a in t[1:])})"
    return t.lstrip("?")


def show_theta(theta):
    if theta is None:
        return "FAIL"
    return "{" + ", ".join(f"{show(v)}/{show(substitute(v, theta))}"
                           for v in theta) + "}"


def show_clause(clause):
    if not clause:
        return "[] (empty clause)"
    return " v ".join(("" if sign else "~") + show(atom) for sign, atom in clause)


def rename(clause, n):
    """Standardise variables apart by attaching a unique suffix."""
    def r(t):
        if is_var(t):
            return f"{t}{n}"
        return tuple(r(a) for a in t) if isinstance(t, tuple) else t
    return [(sign, r(atom)) for sign, atom in clause]


def refute(kb, goal, limit=15):
    """Linear resolution refutation: repeatedly resolve the first literal
    of the current goal clause with a complementary literal of a KB clause."""
    counter = [0]

    def solve(goal, depth, proof):
        if not goal:
            return proof                          # empty clause derived
        if depth == 0:
            return None
        sign, atom = goal[0]
        for idx, clause in enumerate(kb, 1):
            counter[0] += 1
            clause = rename(clause, counter[0])
            for k, (sign2, atom2) in enumerate(clause):
                if sign2 == sign:
                    continue                      # need opposite signs
                theta = unify(atom, atom2)
                if theta is None:
                    continue
                rest = goal[1:] + clause[:k] + clause[k + 1:]
                resolvent = []
                for s, a in rest:
                    lit = (s, substitute(a, theta))
                    if lit not in resolvent:
                        resolvent.append(lit)
                found = solve(resolvent, depth - 1,
                              proof + [(idx, theta, resolvent)])
                if found is not None:
                    return found
        return None

    return solve(goal, limit, [])


if __name__ == "__main__":
    print("PART A: Unification")
    pairs = [
        (("Knows", "John", "?x"), ("Knows", "John", "Jane")),
        (("Knows", "John", "?x"), ("Knows", "?y", "Bill")),
        (("Knows", "John", "?x"), ("Knows", "?y", ("Mother", "?y"))),
        (("P", "?x", ("f", "?y")), ("P", ("g", "?z"), ("f", "A"))),
        (("Knows", "John", "?x"), ("Knows", "?x", "Elizabeth")),
        (("P", "?x"), ("P", ("f", "?x"))),
    ]
    for a, b in pairs:
        print(f"   {show(a):<18} , {show(b):<22} => {show_theta(unify(a, b))}")

    print("\nPART B: Resolution refutation (Colonel West is a criminal)")
    kb = [
        [(False, ("American", "?x")), (False, ("Weapon", "?y")),
         (False, ("Sells", "?x", "?y", "?z")), (False, ("Hostile", "?z")),
         (True, ("Criminal", "?x"))],
        [(False, ("Missile", "?x")), (False, ("Owns", "Nono", "?x")),
         (True, ("Sells", "West", "?x", "Nono"))],
        [(False, ("Missile", "?x")), (True, ("Weapon", "?x"))],
        [(False, ("Enemy", "?x", "America")), (True, ("Hostile", "?x"))],
        [(True, ("Owns", "Nono", "M1"))],
        [(True, ("Missile", "M1"))],
        [(True, ("American", "West"))],
        [(True, ("Enemy", "Nono", "America"))],
    ]
    for i, clause in enumerate(kb, 1):
        print(f"   C{i}: {show_clause(clause)}")
    goal = [(False, ("Criminal", "West"))]
    print(f"   Negated goal G0: {show_clause(goal)}")
    proof = refute(kb, goal)
    for step, (idx, theta, resolvent) in enumerate(proof, 1):
        print(f"   G{step}: resolve G{step - 1} with C{idx} "
              f"=> {show_clause(resolvent)}")
    print("   Empty clause reached: KB entails Criminal(West).")

PART A: Unification
   Knows(John, x)     , Knows(John, Jane)      => {x/Jane}
   Knows(John, x)     , Knows(y, Bill)         => {y/John, x/Bill}
   Knows(John, x)     , Knows(y, Mother(y))    => {y/John, x/Mother(John)}
   P(x, f(y))         , P(g(z), f(A))          => {x/g(z), y/A}
   Knows(John, x)     , Knows(x, Elizabeth)    => FAIL
   P(x)               , P(f(x))                => FAIL

PART B: Resolution refutation (Colonel West is a criminal)
   C1: ~American(x) v ~Weapon(y) v ~Sells(x, y, z) v ~Hostile(z) v Criminal(x)
   C2: ~Missile(x) v ~Owns(Nono, x) v Sells(West, x, Nono)
   C3: ~Missile(x) v Weapon(x)
   C4: ~Enemy(x, America) v Hostile(x)
   C5: Owns(Nono, M1)
   C6: Missile(M1)
   C7: American(West)
   C8: Enemy(Nono, America)
   Negated goal G0: ~Criminal(West)
   G1: resolve G0 with C1 => ~American(West) v ~Weapon(y1) v ~Sells(West, y1, z1) v ~Hostile(z1)
   G2: resolve G1 with C7 => ~Weapon(y1) v ~Sells(West, y1, z1) v ~Hostile(z1)
   G3: resolve G2 with C3 => ~Sell

---
## Experiment 8: Bayes' Theorem and a Simple Bayesian Network for Diagnosis
**Course Outcome:** CO204.4

**Aim:** To apply Bayes' theorem to a medical-test problem and perform inference by enumeration on the burglary-alarm Bayesian network.

**Input:** Part A: P(D) = 0.01, P(+|D) = 0.95, P(+|not D) = 0.05. Part B: the CPTs of the burglary network and eight queries in the list `tests`.


In [8]:
"""Experiment 8: Bayes' theorem and a Bayesian network for diagnosis."""
from itertools import product


# ---------------- Part A: Bayes' theorem (medical test) ----------------
def bayes(prior, sensitivity, false_positive_rate):
    """P(Disease | Positive test)."""
    evidence = sensitivity * prior + false_positive_rate * (1 - prior)
    return sensitivity * prior / evidence, evidence


# ---------------- Part B: Burglary-alarm Bayesian network ----------------
#        Burglary     Earthquake
#              \       /
#                Alarm
#              /       \
#      JohnCalls     MaryCalls
VARS = ["B", "E", "A", "J", "M"]
P_B = 0.001
P_E = 0.002
P_A = {(True, True): 0.95, (True, False): 0.94,
       (False, True): 0.29, (False, False): 0.001}      # P(A=T | B, E)
P_J = {True: 0.90, False: 0.05}                         # P(J=T | A)
P_M = {True: 0.70, False: 0.01}                         # P(M=T | A)


def pr(p_true, value):
    return p_true if value else 1 - p_true


def joint(b, e, a, j, m):
    """Chain rule for the network: product of P(node | parents)."""
    return (pr(P_B, b) * pr(P_E, e) * pr(P_A[(b, e)], a)
            * pr(P_J[a], j) * pr(P_M[a], m))


def query(var, evidence):
    """Inference by enumeration: P(var = True | evidence)."""
    dist = {True: 0.0, False: 0.0}
    for values in product([True, False], repeat=len(VARS)):
        world = dict(zip(VARS, values))
        if all(world[k] == v for k, v in evidence.items()):
            dist[world[var]] += joint(*values)
    return dist[True] / (dist[True] + dist[False])


if __name__ == "__main__":
    print("PART A: Bayes' theorem for a diagnostic test")
    prior, sens, fpr = 0.01, 0.95, 0.05
    print(f"   P(D) = {prior}, P(+|D) = {sens}, P(+|~D) = {fpr}")
    post, evidence = bayes(prior, sens, fpr)
    print(f"   P(+)    = {evidence:.4f}")
    print(f"   P(D|+)  = {post:.4f}")
    post2, _ = bayes(post, sens, fpr)
    print(f"   P(D|++) = {post2:.4f}   (second independent positive test)")

    print("\nPART B: Burglary network, inference by enumeration")
    print(f"   P(J=T, M=T, A=T, B=F, E=F) = "
          f"{joint(False, False, True, True, True):.6f}")
    tests = [
        ("B", {}),
        ("A", {}),
        ("B", {"A": True}),
        ("B", {"J": True}),
        ("B", {"J": True, "M": True}),
        ("E", {"J": True, "M": True}),
        ("A", {"J": True, "M": True}),
        ("B", {"J": True, "M": True, "E": True}),
    ]
    for var, ev in tests:
        given = ", ".join(f"{k}={'T' if v else 'F'}" for k, v in ev.items())
        label = f"P({var}=T | {given})" if given else f"P({var}=T)"
        print(f"   {label:<26} = {query(var, ev):.4f}")

PART A: Bayes' theorem for a diagnostic test
   P(D) = 0.01, P(+|D) = 0.95, P(+|~D) = 0.05
   P(+)    = 0.0590
   P(D|+)  = 0.1610
   P(D|++) = 0.7848   (second independent positive test)

PART B: Burglary network, inference by enumeration
   P(J=T, M=T, A=T, B=F, E=F) = 0.000628
   P(B=T)                     = 0.0010
   P(A=T)                     = 0.0025
   P(B=T | A=T)               = 0.3736
   P(B=T | J=T)               = 0.0163
   P(B=T | J=T, M=T)          = 0.2842
   P(E=T | J=T, M=T)          = 0.1761
   P(A=T | J=T, M=T)          = 0.7607
   P(B=T | J=T, M=T, E=T)     = 0.0033


---
## Experiment 9: Decision-Tree Classifier (ID3) and the Rules Learned
**Course Outcome:** CO204.5

**Aim:** To build a decision tree with the ID3 algorithm (entropy and information gain) on the Play-Tennis dataset and interpret the IF-THEN rules it learns.

**Input:** The 14-example Play-Tennis dataset `DATA` with attributes Outlook, Temperature, Humidity, Wind, and one unseen day to classify.


In [9]:
"""Experiment 9: Decision-tree classifier (ID3) on the Play-Tennis dataset."""
from collections import Counter
from math import log2

ATTRIBUTES = ["Outlook", "Temperature", "Humidity", "Wind"]
DATA = [  # Outlook, Temperature, Humidity, Wind, PlayTennis
    ("Sunny", "Hot", "High", "Weak", "No"),
    ("Sunny", "Hot", "High", "Strong", "No"),
    ("Overcast", "Hot", "High", "Weak", "Yes"),
    ("Rain", "Mild", "High", "Weak", "Yes"),
    ("Rain", "Cool", "Normal", "Weak", "Yes"),
    ("Rain", "Cool", "Normal", "Strong", "No"),
    ("Overcast", "Cool", "Normal", "Strong", "Yes"),
    ("Sunny", "Mild", "High", "Weak", "No"),
    ("Sunny", "Cool", "Normal", "Weak", "Yes"),
    ("Rain", "Mild", "Normal", "Weak", "Yes"),
    ("Sunny", "Mild", "Normal", "Strong", "Yes"),
    ("Overcast", "Mild", "High", "Strong", "Yes"),
    ("Overcast", "Hot", "Normal", "Weak", "Yes"),
    ("Rain", "Mild", "High", "Strong", "No"),
]


def entropy(rows):
    counts = Counter(r[-1] for r in rows)
    return -sum(c / len(rows) * log2(c / len(rows)) for c in counts.values())


def information_gain(rows, col):
    remainder = 0.0
    for value in set(r[col] for r in rows):
        subset = [r for r in rows if r[col] == value]
        remainder += len(subset) / len(rows) * entropy(subset)
    return entropy(rows) - remainder


def id3(rows, columns, depth=0):
    """Return a leaf label or a node {attribute: {value: subtree}}."""
    labels = [r[-1] for r in rows]
    if len(set(labels)) == 1:                  # pure node
        return labels[0]
    if not columns:                            # no attribute left
        return Counter(labels).most_common(1)[0][0]
    gains = {c: information_gain(rows, c) for c in columns}
    best = max(columns, key=lambda c: gains[c])
    if depth == 0:
        print("Information gain at the root:")
        for c in columns:
            print(f"   Gain(S, {ATTRIBUTES[c]:<11}) = {gains[c]:.4f}")
        print(f"   => split on {ATTRIBUTES[best]}\n")
    node = {ATTRIBUTES[best]: {}}
    for value in sorted(set(r[best] for r in rows)):
        subset = [r for r in rows if r[best] == value]
        remaining = [c for c in columns if c != best]
        node[ATTRIBUTES[best]][value] = id3(subset, remaining, depth + 1)
    return node


def show_tree(node, indent="   "):
    if not isinstance(node, dict):
        print(f"{indent}-> {node}")
        return
    (attribute, branches), = node.items()
    for value, child in branches.items():
        print(f"{indent}[{attribute} = {value}]")
        show_tree(child, indent + "    ")


def extract_rules(node, conditions=()):
    if not isinstance(node, dict):
        yield conditions, node
        return
    (attribute, branches), = node.items()
    for value, child in branches.items():
        yield from extract_rules(child, conditions + ((attribute, value),))


def predict(node, sample):
    while isinstance(node, dict):
        (attribute, branches), = node.items()
        node = branches[sample[attribute]]
    return node


if __name__ == "__main__":
    print(f"Training examples: {len(DATA)}   "
          f"Entropy(S) = {entropy(DATA):.4f}\n")
    tree = id3(DATA, list(range(len(ATTRIBUTES))))
    print("Decision tree learned:")
    show_tree(tree)

    print("\nRules read from the tree:")
    for i, (conds, label) in enumerate(extract_rules(tree), 1):
        lhs = " AND ".join(f"{a} = {v}" for a, v in conds)
        print(f"   R{i}: IF {lhs} THEN PlayTennis = {label}")

    correct = sum(predict(tree, dict(zip(ATTRIBUTES, r))) == r[-1] for r in DATA)
    print(f"\nTraining accuracy: {correct}/{len(DATA)}")
    new_day = {"Outlook": "Sunny", "Temperature": "Cool",
               "Humidity": "High", "Wind": "Strong"}
    print("New sample :", new_day)
    print("Prediction : PlayTennis =", predict(tree, new_day))

Training examples: 14   Entropy(S) = 0.9403

Information gain at the root:
   Gain(S, Outlook    ) = 0.2467
   Gain(S, Temperature) = 0.0292
   Gain(S, Humidity   ) = 0.1518
   Gain(S, Wind       ) = 0.0481
   => split on Outlook

Decision tree learned:
   [Outlook = Overcast]
       -> Yes
   [Outlook = Rain]
       [Wind = Strong]
           -> No
       [Wind = Weak]
           -> Yes
   [Outlook = Sunny]
       [Humidity = High]
           -> No
       [Humidity = Normal]
           -> Yes

Rules read from the tree:
   R1: IF Outlook = Overcast THEN PlayTennis = Yes
   R2: IF Outlook = Rain AND Wind = Strong THEN PlayTennis = No
   R3: IF Outlook = Rain AND Wind = Weak THEN PlayTennis = Yes
   R4: IF Outlook = Sunny AND Humidity = High THEN PlayTennis = No
   R5: IF Outlook = Sunny AND Humidity = Normal THEN PlayTennis = Yes

Training accuracy: 14/14
New sample : {'Outlook': 'Sunny', 'Temperature': 'Cool', 'Humidity': 'High', 'Wind': 'Strong'}
Prediction : PlayTennis = No


---
## Experiment 10: Mini-Project: Rule-Based Expert System and a Basic Chatbot
**Course Outcome:** CO204.5

**Aim:** To develop (a) a rule-based expert system that identifies an animal by forward chaining with an explanation facility and (b) a basic pattern-matching chatbot.

**Input:** Part A: four cases, each a set of observed features. Part B: seven user messages. Replace the fixed lists with `input()` in a loop to make it interactive.


In [10]:
"""Experiment 10 (Mini-project): Rule-based expert system and a basic chatbot.
Part A: animal-identification expert system (forward chaining + explanation).
Part B: pattern-matching helpdesk chatbot."""
import re

# ---------------- Part A: expert system ----------------
RULES = [  # (rule id, IF all of these facts, THEN conclude this fact)
    ("R1", {"has hair"}, "is a mammal"),
    ("R2", {"gives milk"}, "is a mammal"),
    ("R3", {"has feathers"}, "is a bird"),
    ("R4", {"flies", "lays eggs"}, "is a bird"),
    ("R5", {"is a mammal", "eats meat"}, "is a carnivore"),
    ("R6", {"is a mammal", "has hooves"}, "is an ungulate"),
    ("R7", {"is a carnivore", "has tawny colour", "has dark spots"},
     "is a CHEETAH"),
    ("R8", {"is a carnivore", "has tawny colour", "has black stripes"},
     "is a TIGER"),
    ("R9", {"is an ungulate", "has long neck", "has dark spots"},
     "is a GIRAFFE"),
    ("R10", {"is an ungulate", "has black stripes"}, "is a ZEBRA"),
    ("R11", {"is a bird", "cannot fly", "swims"}, "is a PENGUIN"),
    ("R12", {"is a bird", "flies well"}, "is an ALBATROSS"),
]


def infer(observations):
    """Forward chaining: fire rules until working memory stops growing."""
    memory, fired = set(observations), []
    changed = True
    while changed:
        changed = False
        for rule_id, conditions, conclusion in RULES:
            if conditions <= memory and conclusion not in memory:
                memory.add(conclusion)
                fired.append((rule_id, conditions, conclusion))
                changed = True
    return memory, fired


def consult(name, observations):
    print(f"Case: {name}")
    print("   Observed :", ", ".join(sorted(observations)))
    memory, fired = infer(observations)
    for rule_id, conditions, conclusion in fired:       # explanation facility
        print(f"   {rule_id:<3} fired: {' AND '.join(sorted(conditions))} "
              f"=> {conclusion}")
    verdict = [f for f in memory if f.split()[-1].isupper()]
    print("   Diagnosis:", verdict[0] if verdict else "animal not identified")


# ---------------- Part B: chatbot ----------------
PATTERNS = [
    (r"\b(hi|hello|hey|namaste)\b",
     "Hello! I am AI-Lab Bot. Ask me about the AI laboratory."),
    (r"\bmy name is (\w+)", "Nice to meet you, {0}!"),
    (r"\b(timing|time|when)\b",
     "The AI lab runs for 2 hours every week as per your batch timetable."),
    (r"\b(experiments?|practicals?)\b",
     "There are 10 experiments: search, games, CSP, logic, Bayes, "
     "decision trees and a mini-project."),
    (r"\b(language|python|prolog)\b",
     "We use Python 3. Experiment 6 may also be done in Prolog."),
    (r"\b(book|textbook|reference)\b",
     "Refer to Russell and Norvig, 'Artificial Intelligence: A Modern "
     "Approach'."),
    (r"\bi am (stuck|confused|lost)\b",
     "Do not worry about being {0}. Trace the algorithm by hand first."),
    (r"\b(thanks|thank you)\b", "You are welcome. Happy learning!"),
    (r"\b(bye|exit|quit)\b", "Goodbye!"),
]
DEFAULT = "Sorry, I did not understand. Try asking about experiments or timing."


def reply(message):
    for pattern, answer in PATTERNS:
        found = re.search(pattern, message, re.IGNORECASE)
        if found:
            return answer.format(*found.groups())
    return DEFAULT


if __name__ == "__main__":
    print("PART A: Animal identification expert system")
    consult("Animal 1", {"has hair", "eats meat", "has tawny colour",
                         "has black stripes"})
    consult("Animal 2", {"has feathers", "cannot fly", "swims"})
    consult("Animal 3", {"gives milk", "has hooves", "has long neck",
                         "has dark spots"})
    consult("Animal 4", {"has scales", "swims"})

    print("\nPART B: Chatbot session")
    for message in ["Hello", "My name is Asha", "How many experiments are there?",
                    "Which language do we use?", "I am stuck in A* search",
                    "What is the weather today?", "Thanks, bye"]:
        print("   You :", message)
        print("   Bot :", reply(message))

PART A: Animal identification expert system
Case: Animal 1
   Observed : eats meat, has black stripes, has hair, has tawny colour
   R1  fired: has hair => is a mammal
   R5  fired: eats meat AND is a mammal => is a carnivore
   R8  fired: has black stripes AND has tawny colour AND is a carnivore => is a TIGER
   Diagnosis: is a TIGER
Case: Animal 2
   Observed : cannot fly, has feathers, swims
   R3  fired: has feathers => is a bird
   R11 fired: cannot fly AND is a bird AND swims => is a PENGUIN
   Diagnosis: is a PENGUIN
Case: Animal 3
   Observed : gives milk, has dark spots, has hooves, has long neck
   R2  fired: gives milk => is a mammal
   R6  fired: has hooves AND is a mammal => is an ungulate
   R9  fired: has dark spots AND has long neck AND is an ungulate => is a GIRAFFE
   Diagnosis: is a GIRAFFE
Case: Animal 4
   Observed : has scales, swims
   Diagnosis: animal not identified

PART B: Chatbot session
   You : Hello
   Bot : Hello! I am AI-Lab Bot. Ask me about the AI lab